# Check the workflow before training

The runner executes these short checks after batching and before training. They use the actual dataset and model. No temporary test model or test class is needed.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import gc
import numpy as np
import torch

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())

if "example_batch" not in globals():
    run_notebook("03_batching.ipynb")

## Check split boundaries

Training and validation must not share paths or transcript groups. The original test rows must stay unchanged.

In [ ]:
assert set(train_df["audio_path"]).isdisjoint(validation_df["audio_path"])
assert set(train_df["group"]).isdisjoint(validation_df["group"])
assert set(original_train["audio_path"]).isdisjoint(test_df["audio_path"])
assert test_df["audio_path"].tolist() == original_test["audio_path"].tolist()
assert len(train_df) + len(validation_df) == len(original_train)
print("Split checks passed.")

## Check settings and model inputs

The input should have 80 Mel bins and 3,000 time frames. Every model parameter must be trainable. The example loss must be finite.

In [ ]:
assert EPOCHS >= 1 and BATCH_SIZE >= 1 and ACCUMULATION_STEPS >= 1
assert LEARNING_RATE > 0 and 0 < VALIDATION_FRACTION < 1
assert RUN_DIR != DATA_DIR and not RUN_DIR.is_relative_to(DATA_DIR)
assert tuple(example_batch["input_features"].shape[1:]) == (80, 3000)
assert example_batch["labels"].shape[1] <= model.config.max_target_positions
assert np.isfinite(example_loss)
assert all(parameter.requires_grad for parameter in model.parameters())
print("Settings, batch, and full-model checks passed.")

# Free the example batch before starting the larger training loop.
del example_batch
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()